In [1]:
import sys
sys.path.insert(0, "src")
import pandas as pd
from rag.pipeline import RAGPipeline

In [3]:
pipeline = RAGPipeline()
pipeline.build_index("policies")
print(f"Indexed {pipeline.store.index.ntotal} chunks from "
      f"{len(set(c.source for c in pipeline.store.chunks))} documents")

Indexed 23 chunks from 6 documents


In [4]:
EVAL_SET = [
    {"id": "Q1", "query": "What is the minimum FICO score for standard loan approval?",
     "expected_source": "credit_policy.pdf"},
    {"id": "Q2", "query": "What DTI ratio requires senior underwriter sign-off?",
     "expected_source": "credit_policy.pdf"},
    {"id": "Q3", "query": "What probability of default classifies an applicant as high risk?",
     "expected_source": "risk_policy.pdf"},
    {"id": "Q4", "query": "What are the criteria for a loan to be automatically approved?",
     "expected_source": "approval_policy.pdf"},
    {"id": "Q5", "query": "What is the maximum loan-to-value ratio for a new vehicle loan?",
     "expected_source": "collateral_policy.pdf"},
    {"id": "Q6", "query": "What compensating factors allow a subprime applicant to be approved?",
     "expected_source": "exception_policy.pdf"},
    {"id": "Q7", "query": "What interest rate range applies to Risk Grade C loans?",
     "expected_source": "lending_policy.pdf"},
    {"id": "Q8", "query": "How often must the credit risk model be revalidated?",
     "expected_source": "risk_policy.pdf"},
    {"id": "Q9", "query": "What valuation is required for real property collateral over $150,000?",
     "expected_source": "collateral_policy.pdf"},
    {"id": "Q10", "query": "What is the Bank's policy on cryptocurrency day-trading commissions?",
     "expected_source": None}
]
len(EVAL_SET)

10

In [5]:
def evaluate_retrieval(pipeline, eval_set, k=3):
    rows = []
    for case in eval_set:
        results = pipeline.retrieve(case["query"], k=k)
        retrieved_sources = [chunk.source for chunk, _ in results]
        top_score = results[0][1] if results else 0.0

        if case["expected_source"] is None:
            rows.append({
                "id": case["id"], "query": case["query"], "expected": "(out of scope)",
                "top_retrieved": retrieved_sources[0] if retrieved_sources else None,
                "top_score": round(top_score, 3), "hit": None, "rank": None,
            })
            continue

        hit = case["expected_source"] in retrieved_sources
        rank = retrieved_sources.index(case["expected_source"]) + 1 if hit else None
        rows.append({
            "id": case["id"], "query": case["query"], "expected": case["expected_source"],
            "top_retrieved": retrieved_sources[0] if retrieved_sources else None,
            "top_score": round(top_score, 3), "hit": hit, "rank": rank,
        })
    return pd.DataFrame(rows)

results_df = evaluate_retrieval(pipeline, EVAL_SET, k=3)
results_df

,id,query,expected,top_retrieved,top_score,hit,rank
0,Q1,What is the minimum FICO score for standard lo...,credit_policy.pdf,credit_policy.pdf,0.799,True,1.0
1,Q2,What DTI ratio requires senior underwriter sig...,credit_policy.pdf,credit_policy.pdf,0.841,True,1.0
2,Q3,What probability of default classifies an appl...,risk_policy.pdf,risk_policy.pdf,0.756,True,1.0
3,Q4,What are the criteria for a loan to be automat...,approval_policy.pdf,approval_policy.pdf,0.792,True,1.0
4,Q5,What is the maximum loan-to-value ratio for a ...,collateral_policy.pdf,collateral_policy.pdf,0.789,True,1.0
5,Q6,What compensating factors allow a subprime app...,exception_policy.pdf,exception_policy.pdf,0.779,True,1.0
6,Q7,What interest rate range applies to Risk Grade...,lending_policy.pdf,lending_policy.pdf,0.753,True,1.0
7,Q8,How often must the credit risk model be revali...,risk_policy.pdf,risk_policy.pdf,0.768,True,1.0
8,Q9,What valuation is required for real property c...,collateral_policy.pdf,collateral_policy.pdf,0.738,True,1.0
9,Q10,What is the Bank's policy on cryptocurrency da...,(out of scope),lending_policy.pdf,0.668,None,NaN


In [6]:
scored = results_df[results_df['hit'].notna()]
hit_rate = scored['hit'].mean()
mrr = (1 / scored['rank']).fillna(0).mean()

print(f"Hit Rate@3: {hit_rate:.1%}  ({scored['hit'].sum()}/{len(scored)} queries)")
print(f"MRR@3:      {mrr:.3f}")

oos_score = results_df.loc[results_df['expected'] == '(out of scope)', 'top_score'].iloc[0]
in_scope_avg_score = scored['top_score'].mean()
print(f"\nOut-of-scope top score: {oos_score:.3f}  vs. in-scope average: {in_scope_avg_score:.3f}")

Hit Rate@3: 100.0%  (9/9 queries)
MRR@3:      1.000

Out-of-scope top score: 0.668  vs. in-scope average: 0.779


In [7]:
misses = results_df[results_df['hit'] == False]
if len(misses):
    for _, row in misses.iterrows():
        print(f"[{row['id']}] {row['query']}")
        print(f"  expected: {row['expected']}  |  got: {row['top_retrieved']}")
else:
    print("No misses at k=3 on this evaluation set.")

No misses at k=3 on this evaluation set.


In [8]:
import os

if os.environ.get("ANTHROPIC_API_KEY"):
    result = pipeline.answer(
        question="Why can't this applicant be automatically approved?",
        pd_score=0.28,
        risk_drivers={"dti": 0.18, "fico_range_low": -0.09, "revol_util": 0.11},
        k=4,
    )
    print(result["answer"])
    print()
    print("Sources used:")
    for s in result["retrieved_sources"]:
        print(f"  - {s['source']} (score={s['score']:.3f})")
else:
    print("ANTHROPIC_API_KEY not set — skipping live generation call. "
          "Retrieval-only evaluation above does not require an API key.")

ANTHROPIC_API_KEY not set — skipping live generation call. Retrieval-only evaluation above does not require an API key.
